### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps transformers==5.5.0
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [2]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

### Unsloth

In [3]:
from unsloth import FastLanguageModel
import torch

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it",
    max_seq_length = 2048,
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)

# Note: Using 'tokenizer' instead of 'processor' for language models
processor = tokenizer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

In [4]:
!pip install --upgrade torchao

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
    target_modules = "all-linear",
)

<a name="Data"></a>
### Data Prep
Use Amod mental health counseling conversations

In [5]:
from datasets import load_dataset
dataset = load_dataset("Amod/mental_health_counseling_conversations", split = "train")

README.md: 0.00B [00:00, ?B/s]

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

Dataset overview

In [6]:
dataset

Dataset({
    features: ['Context', 'Response'],
    num_rows: 3512
})

In [7]:
dataset[2]["Context"]

"I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?"

In [8]:
dataset[2]["Response"]

"First thing I'd suggest is getting the sleep you need or it will impact how you think and feel. I'd look at finding what is going well in your life and what you can be grateful for. I believe everyone has talents and wants to find their purpose in life. I think you can figure it out with some help."

In [9]:
def format_example(row):
    convo = [
        {"role": "system",
         "content": "You are a warm, empathetic companion..."},
        {"role": "user",    "content": row["Context"]},
        {"role": "assistant","content": row["Response"]},
    ]
    return {"text": processor.apply_chat_template(
        convo, tokenize=False, add_generation_prompt=False)}

dataset = dataset.map(format_example)

# 95/5 train/validation split for proper evaluation
dataset = dataset.train_test_split(test_size=0.05, seed=3407)
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['test'])}")

Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

The first example is now structured like below:

In [10]:
dataset[0]

{'Context': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?",
 'Response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is someh

Lets take the Gemma 4 instruction chat template and use it in our base model

In [11]:
from unsloth import get_chat_template

processor = get_chat_template(
    processor,
    "gemma-4"
)

Before fine-tuning, let us evaluate the base model's performance.

In [12]:
text = dataset[2]["Context"]
instruction = "You are a warm, empathetic companion."

messages = [
    {
        "role": "user",
        "content": [{"type": "text", "text": f"{instruction}\n\n{text}"}],
    }
]
input_text = processor.apply_chat_template(messages, add_generation_prompt = True)
inputs = processor(
    text = [input_text],
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer

text_streamer = TextStreamer(processor, skip_prompt = True)
result = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                        use_cache = True, temperature = 1.0, top_p = 0.95, top_k = 64)

It takes a lot of courage to reach out and share what you're going through, and I want you to know that I hear you. Please know that what you are feeling right now—this heavy, overwhelming sense of worthlessness—is incredibly painful, and you do not have to carry it alone. I am here to sit with you in this darkness, and I want to help you find a little bit of light, even if it feels distant right now.

First, I need to be very clear about something, because your safety is the most important thing right now.

**If at any point these feelings become more intense


<a name="Train"></a>
### Train the model
We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!


In [13]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset["train"],
    eval_dataset  = dataset["test"],
    processing_class = processor.tokenizer,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        max_grad_norm = 0.3,
        warmup_steps = 25,          # ~3% of total steps across 2 epochs
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 50,
        save_strategy = "steps",
        save_steps = 50,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",

        dataset_text_field = "text",
        max_length = 2048,
    )
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/3512 [00:00<?, ? examples/s]

In [14]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
10.662 GB of memory reserved.


In [15]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,512 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 82,444,288 of 8,078,600,736 (1.02% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.713696
2,2.482602
3,2.118168
4,1.738227
5,1.485828
6,1.332540
7,1.188127
8,1.277528
9,1.153783
10,1.047246


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [16]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

505.5808 seconds used for training.
8.43 minutes used for training.
Peak reserved memory = 13.422 GB.
Peak reserved memory for training = 2.76 GB.
Peak reserved memory % of max memory = 92.165 %.
Peak reserved memory for training % of max memory = 18.952 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can modify the instruction and input—just leave the output blank.

We'll use the best hyperparameters for inference on Gemma: `top_p=0.95`, `top_k=64`, and `temperature=1.0`.

In [17]:
image = dataset[9]["text"]
instruction = "I feel peer pressured"

messages = [
    {
        "role": "user",
        "content": [{"type": "text", "text": instruction}],
    }
]

input_text = processor.apply_chat_template(messages, add_generation_prompt = True)
inputs = processor(
    text = [input_text],
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer

text_streamer = TextStreamer(processor, skip_prompt = True)
result = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 512,
                        use_cache = True, temperature = 1.0, top_p = 0.95, top_k = 64)

It sounds like you're feeling pressured by your friends. It's important to remember that you have the right to say no to things you don't want to do, even if your friends are doing them. You don't have to do anything that you don't want to do just because your friends are.

If you feel like your friends are pressuring you to do things you don't want to do, you can tell them that you don't want to do it. You can also tell them that you don't like it and that you don't want to do it. If they keep pressuring you, you can tell them that you are not going to do it.

If you feel like your friends are being mean to you or are trying to make you feel bad about yourself, you can tell them that you don't like it. You can also tell them that you won't do it. If they keep being mean to you, you can walk away from them.

Remember that you are in charge of what you do with your life. You don't have to do anything that you don't want to do just because your friends are doing it. You have the right to

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, use Hugging Face’s `push_to_hub` for online saving, or `save_pretrained` for local storage.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [22]:
import os
from google.colab import userdata
from huggingface_hub import HfApi

# Securely fetch your token from Colab Secrets
HUGGINGFACE_TOKEN = userdata.get('HF_TOKEN')

# Get your Hugging Face username automatically
api = HfApi(token=HUGGINGFACE_TOKEN)
username = api.whoami()["name"]
repo_id = f"{username}/gemma_4_lora"

print(f"Pushing to: {repo_id}")

# Push the LoRA adapters and processor
model.push_to_hub(repo_id, token = HUGGINGFACE_TOKEN)
processor.push_to_hub(repo_id, token = HUGGINGFACE_TOKEN)

Pushing to: nglong14/gemma_4_lora


README.md:   0%|          | 0.00/570 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  560kB /  330MB            

Saved model to https://huggingface.co/nglong14/gemma_4_lora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmps2m0nvp2/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mps2m0nvp2/tokenizer.json: 100%|##########| 32.2MB / 32.2MB            